# Holiday Calendar (Idiomatic Notebook)

This notebook uses the well-supported `holidays` package to generate holiday dates with a concise, notebook-first workflow.

In [1]:
# Notebook parameters
start_year = 2024
end_year = 2028
country = "US"
subdiv = None  # Example: "CA" for California

In [2]:
# Install once per environment if needed
%pip -q install holidays ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import pandas as pd
import holidays

In [4]:
# Build a tidy holiday table
years = range(start_year, end_year + 1)
cal = holidays.country_holidays(country, subdiv=subdiv, years=years, observed=True)

df = (
    pd.Series(cal, name="Holiday")
    .rename_axis("Date")
    .sort_index()
    .reset_index()
)

df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Weekday"] = df["Date"].dt.day_name()

df.head(10)

,Date,Holiday,Year,Weekday
0,2024-01-01,New Year's Day,2024,Monday
1,2024-01-15,Martin Luther King Jr. Day,2024,Monday
2,2024-02-19,Washington's Birthday,2024,Monday
3,2024-05-27,Memorial Day,2024,Monday
4,2024-06-19,Juneteenth National Independence Day,2024,Wednesday
5,2024-07-04,Independence Day,2024,Thursday
6,2024-09-02,Labor Day,2024,Monday
7,2024-10-14,Columbus Day,2024,Monday
8,2024-11-11,Veterans Day,2024,Monday
9,2024-11-28,Thanksgiving Day,2024,Thursday


In [5]:
# Compact yearly summary
summary = (
    df.groupby("Year", as_index=False)
    .agg(Holiday_Count=("Holiday", "count"))
)
summary

,Year,Holiday_Count
0,2024,11
1,2025,11
2,2026,12
3,2027,15
4,2028,12


In [6]:
# Inspect one year
year = 2026
df.loc[df["Year"] == year, ["Date", "Weekday", "Holiday"]].reset_index(drop=True)

,Date,Weekday,Holiday
0,2026-01-01,Thursday,New Year's Day
1,2026-01-19,Monday,Martin Luther King Jr. Day
2,2026-02-16,Monday,Washington's Birthday
3,2026-05-25,Monday,Memorial Day
4,2026-06-19,Friday,Juneteenth National Independence Day
5,2026-07-03,Friday,Independence Day (observed)
6,2026-07-04,Saturday,Independence Day
7,2026-09-07,Monday,Labor Day
8,2026-10-12,Monday,Columbus Day
9,2026-11-11,Wednesday,Veterans Day


## Quick Validation Tests

Each test is a single code cell that shows expected vs actual values as notebook output.
Assertions enforce correctness without print-based status messages.

In [7]:
# Approval testing helpers imported from shared utility
import importlib
import approval_test_utils as atu
importlib.reload(atu)

configure_approval_store = atu.configure_approval_store
to_iso_records = atu.to_iso_records
show_approval_test = atu.show_approval_test
approval_from_dataframe = atu.approval_from_dataframe

In [8]:
# APPROVAL_CONFIG
TEST_NOTEBOOK_PATH = "/workspaces/coding/holiday_calculator_tested.ipynb"
APPROVALS_NOTEBOOK_PATH = None  # Optional override path for approvals notebook
configure_approval_store(
    test_notebook_path=TEST_NOTEBOOK_PATH,
    approvals_notebook_path=APPROVALS_NOTEBOOK_PATH,
)

In [9]:
# Test 1: required columns are present (one-cell test)
actual_columns = sorted(df.columns.tolist())
required = ["Date", "Holiday", "Weekday", "Year"]
missing = [c for c in required if c not in actual_columns]
assert not missing, f"Missing columns: {missing}"

show_approval_test(
    test_id="required_columns_present",
    description="Dataframe includes required columns.",
    actual={"columns": actual_columns},
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "required_columns_present",
  "description": "Dataframe includes required columns.",
  "value": {
    "columns": [
      "Date",
      "Holiday",
      "Weekday",
      "Year"
    ]
  },
  "expectedValue": {
    "columns": [
      "Date",
      "Holiday",
      "Weekday",
      "Year"
    ]
  },
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    },
    {
      "@type": "PropertyValue",
      "name": "approvalsNotebook",
      "value": "/workspaces/coding/__approvals__/holiday_calculator_tested.ipynb"
    }
  ]
}

In [10]:
# Test 2: year boundaries are correct (one-cell test)
actual_min = int(df["Year"].min())
actual_max = int(df["Year"].max())
assert actual_min == start_year, f"Start year mismatch: expected {start_year}, got {actual_min}"
assert actual_max == end_year, f"End year mismatch: expected {end_year}, got {actual_max}"

show_approval_test(
    test_id="year_boundaries",
    description="Observed year range matches configured notebook parameters.",
    actual={"startYear": actual_min, "endYear": actual_max},
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "year_boundaries",
  "description": "Observed year range matches configured notebook parameters.",
  "value": {
    "startYear": 2024,
    "endYear": 2028
  },
  "expectedValue": {
    "startYear": 2024,
    "endYear": 2028
  },
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    },
    {
      "@type": "PropertyValue",
      "name": "approvalsNotebook",
      "value": "/workspaces/coding/__approvals__/holiday_calculator_tested.ipynb"
    }
  ]
}

In [11]:
# Test 3: date ordering and weekday derivation (one-cell test)
is_sorted = bool(df["Date"].is_monotonic_increasing)
weekday_match = bool((df["Date"].dt.day_name() == df["Weekday"]).all())
assert is_sorted, "Dates are not sorted ascending"
assert weekday_match, "Weekday values do not match Date"

show_approval_test(
    test_id="date_order_and_weekday_derivation",
    description="Dates are sorted and weekday text matches derived weekday names.",
    actual={"datesSorted": is_sorted, "weekdayMatchesDate": weekday_match},
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "date_order_and_weekday_derivation",
  "description": "Dates are sorted and weekday text matches derived weekday names.",
  "value": {
    "datesSorted": true,
    "weekdayMatchesDate": true
  },
  "expectedValue": {
    "datesSorted": true,
    "weekdayMatchesDate": true
  },
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    },
    {
      "@type": "PropertyValue",
      "name": "approvalsNotebook",
      "value": "/workspaces/coding/__approvals__/holiday_calculator_tested.ipynb"
    }
  ]
}

In [12]:
# Test 4: known holiday spot checks (one-cell test)
known = pd.DataFrame([
    {"Date": pd.Timestamp("2024-01-01"), "Expected": "New Year's Day"},
    {"Date": pd.Timestamp("2024-11-28"), "Expected": "Thanksgiving Day"},
    {"Date": pd.Timestamp("2026-05-25"), "Expected": "Memorial Day"},
])

actual = (
    df.merge(known[["Date"]], on="Date", how="inner")
      .groupby("Date", as_index=False)
      .agg(Expected=("Holiday", lambda s: " | ".join(sorted(set(s)))))
)
actual_records = to_iso_records(actual)

show_approval_test(
    test_id="known_holiday_spot_checks",
    description="Known holiday checkpoints match expected names for specific dates.",
    actual=actual_records,
    sort_by=["Date", "Expected"],
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "known_holiday_spot_checks",
  "description": "Known holiday checkpoints match expected names for specific dates.",
  "value": [
    {
      "Date": "2024-01-01",
      "Expected": "New Year's Day"
    },
    {
      "Date": "2024-11-28",
      "Expected": "Thanksgiving Day"
    },
    {
      "Date": "2026-05-25",
      "Expected": "Memorial Day"
    }
  ],
  "expectedValue": [
    {
      "Date": "2024-01-01",
      "Expected": "New Year's Day"
    },
    {
      "Date": "2024-11-28",
      "Expected": "Thanksgiving Day"
    },
    {
      "Date": "2026-05-25",
      "Expected": "Memorial Day"
    }
  ],
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    },
    {
      "@type": "PropertyValue",
      "name": "approvalsNotebook",
      "value": "/workspaces/coding/__approvals__/holiday_calculator_tested.ipynb"
    }
  ]
}

## Concise Approval Tests (Visible Approved vs Actual)

This style keeps approvals directly in notebook cells.
Each test cell displays Approved and Actual values, then fails with a focused diff when they differ.

### Notebook Testing Patterns In Practice

- ipytest: run pytest directly inside notebook cells via magics.
- nbval or pytest-notebook: rerun whole notebooks and compare stored outputs, usually in CI.
- testbook: write tests in separate Python files against notebook code.
- ApprovalTests.Python: classic approved vs received workflow with diff tools.

This notebook uses per-cell approved tables so each test visibly shows Approved and Actual values in one place.

In [13]:
# Helper note: JSON-LD helpers are defined above the tests to support linear top-to-bottom execution.
# This cell is intentionally left minimal to avoid redefining hooks later in the notebook.

In [14]:
# Approval test 1: full 2026 holiday list (approved baseline stored in storage cell)
actual_2026 = df.loc[df["Year"] == 2026, ["Date", "Holiday"]]
approval_from_dataframe(
    test_id="approval_us_2026_holidays",
    description="Federal holidays for 2026 (including observed).",
    actual_df=actual_2026,
    sort_by=["Date", "Holiday"],
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "approval_us_2026_holidays",
  "description": "Federal holidays for 2026 (including observed).",
  "value": [
    {
      "Date": "2026-01-01",
      "Holiday": "New Year's Day"
    },
    {
      "Date": "2026-01-19",
      "Holiday": "Martin Luther King Jr. Day"
    },
    {
      "Date": "2026-02-16",
      "Holiday": "Washington's Birthday"
    },
    {
      "Date": "2026-05-25",
      "Holiday": "Memorial Day"
    },
    {
      "Date": "2026-06-19",
      "Holiday": "Juneteenth National Independence Day"
    },
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    },
    {
      "Date": "2026-09-07",
      "Holiday": "Labor Day"
    },
    {
      "Date": "2026-10-12",
      "Holiday": "Columbus Day"
    },
    {
      "Date": "2026-11-11",
      "Holiday": "Veterans Day"
    },
    {
      "Date": "2026-11-26",
      "Holiday": "Thanksgiving Day"
    },
    {
      "Date": "2026-12-25",
      "Holiday": "Christmas Day"
    }
  ],
  "expectedValue": [
    {
      "Date": "2026-01-01",
      "Holiday": "New Year's Day"
    },
    {
      "Date": "2026-01-19",
      "Holiday": "Martin Luther King Jr. Day"
    },
    {
      "Date": "2026-02-16",
      "Holiday": "Washington's Birthday"
    },
    {
      "Date": "2026-05-25",
      "Holiday": "Memorial Day"
    },
    {
      "Date": "2026-06-19",
      "Holiday": "Juneteenth National Independence Day"
    },
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    },
    {
      "Date": "2026-09-07",
      "Holiday": "Labor Day"
    },
    {
      "Date": "2026-10-12",
      "Holiday": "Columbus Day"
    },
    {
      "Date": "2026-11-11",
      "Holiday": "Veterans Day"
    },
    {
      "Date": "2026-11-26",
      "Holiday": "Thanksgiving Day"
    },
    {
      "Date": "2026-12-25",
      "Holiday": "Christmas Day"
    }
  ],
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    },
    {
      "@type": "PropertyValue",
      "name": "approvalsNotebook",
      "value": "/workspaces/coding/__approvals__/holiday_calculator_tested.ipynb"
    }
  ]
}

In [15]:
# Approval test 2: Independence Day observed behavior in 2026
actual_observed = df.loc[
    (df["Date"].between(pd.Timestamp("2026-07-03"), pd.Timestamp("2026-07-04")))
    & (df["Holiday"].str.contains("Independence Day")),
    ["Date", "Holiday"],
]
approval_from_dataframe(
    test_id="approval_independence_day_observed_2026",
    description="Observed Independence Day appears on Friday when July 4 is Saturday.",
    actual_df=actual_observed,
    sort_by=["Date", "Holiday"],
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "approval_independence_day_observed_2026",
  "description": "Observed Independence Day appears on Friday when July 4 is Saturday.",
  "value": [
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    }
  ],
  "expectedValue": [
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    }
  ],
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": "Approved"
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    },
    {
      "@type": "PropertyValue",
      "name": "approvalsNotebook",
      "value": "/workspaces/coding/__approvals__/holiday_calculator_tested.ipynb"
    }
  ]
}